Advanced Linear Algebra For Machine Learning

Advanced (but important)
- Matrix Decompositions (SVD, QR, Cholesky)
- Eigenvalues & Eigenvectors
- Rank
- Determinant & Inverse (Deep Dive)

In [ ]:
import math
import numpy as np
import torch as th
import tensorflow as tf
import torch.nn.functional as F

In [ ]:
# Advanced (but important)
# 9.  Matrix Decompositions
# 10. Eigenvalues & Eigenvectors
# 11. Rank
# 12. Determinant & Inverse (Deep Dive)

9. Matrix Decompositions
A decomposition breaks a matrix into simpler, structured pieces — like factoring 12 = 3 × 4, but for matrices.

Why it matters in ML:
- SVD → PCA, recommendation systems, image compression, NLP (LSA)
- QR → Numerically stable least squares, Gram-Schmidt
- Cholesky → Gaussian Processes, sampling from multivariate Gaussians

SVD — Singular Value Decomposition (most important in ML)
Any matrix A can be written as: A = U @ S @ Vᵀ

- U → Left singular vectors (directions in output space)
- S → Diagonal matrix of singular values (importance / strength of each direction)
- Vᵀ → Right singular vectors (directions in input space)

Key insight: singular values tell you which directions carry the most information.
Drop the small ones → compression with minimal information loss.

In [ ]:
# Base Python: SVD is too complex to implement from scratch usefully.
# The key conceptual idea:

A = [[3, 1, 1],
     [-1, 3, 1]]

# SVD splits A into: rotation (U) → scale (S) → rotation (Vᵀ)
# The singular values (diagonal of S) measure "how important" each component is.
# Keeping only the top-k singular values = compression (PCA is built on this idea).
print("SVD decomposes A into U @ S @ Vᵀ")
print(f"A is a {len(A)}x{len(A[0])} matrix")
print("The singular values reveal hidden structure (importance per direction)")

In [ ]:
# NumPy SVD

A = np.array([[3, 1, 1],
              [-1, 3, 1]], dtype=float)

# full_matrices=False → thin/economy SVD (U is m×k, not m×m)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# Reconstruct: A = U @ diag(s) @ Vt
A_reconstructed = U @ np.diag(s) @ Vt

print(f"Original A:\n{A}")
print(f"\nU (left singular vectors, shape {U.shape}):\n{np.round(U, 4)}")
print(f"\nSingular Values s: {np.round(s, 4)}")
print(f"  (First value is largest → most important direction)")
print(f"\nVt (right singular vectors, shape {Vt.shape}):\n{np.round(Vt, 4)}")
print(f"\nReconstructed A = U @ diag(s) @ Vt:\n{np.round(A_reconstructed, 4)}")
print(f"\nPerfect reconstruction: {np.allclose(A, A_reconstructed)}")

In [ ]:
# PyTorch SVD

A_pt = th.tensor([[3.0, 1.0, 1.0],
                  [-1.0, 3.0, 1.0]])

U_pt, s_pt, Vh_pt = th.linalg.svd(A_pt, full_matrices=False)

A_reconstructed_pt = U_pt @ th.diag(s_pt) @ Vh_pt

print(f"PyTorch Singular Values: {s_pt.tolist()}")
print(f"Reconstruction matches: {th.allclose(A_pt, A_reconstructed_pt, atol=1e-5)}")

In [ ]:
# TensorFlow SVD
# Note: TF returns (s, U, V) — V is NOT transposed, unlike NumPy

A_tf = tf.constant([[3.0, 1.0, 1.0],
                    [-1.0, 3.0, 1.0]])

s_tf, U_tf, V_tf = tf.linalg.svd(A_tf)

# Reconstruct: A = U @ diag(s) @ Vᵀ
A_reconstructed_tf = U_tf @ tf.linalg.diag(s_tf) @ tf.transpose(V_tf)

print(f"TF Singular Values: {s_tf.numpy()}")
print(f"Reconstruction matches: {np.allclose(A_tf.numpy(), A_reconstructed_tf.numpy(), atol=1e-5)}")

In [ ]:
# ML Example: Image Compression via SVD
# Keep only top-k singular values → compressed approximation
# This is the mathematical foundation of JPEG compression and PCA

np.random.seed(42)
image = np.random.randint(50, 200, size=(8, 8)).astype(float)

U, s, Vt = np.linalg.svd(image, full_matrices=False)

print(f"Original Image (8x8):\n{image.astype(int)}")
print(f"\nAll Singular Values: {np.round(s, 1)}")
print("(Larger value = more important direction)\n")

for k in [1, 2, 4, 8]:
    compressed = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    mse = np.mean((image - compressed) ** 2)
    storage_ratio = (k * (8 + 1 + 8)) / (8 * 8)
    print(f"Top-{k} components | MSE: {mse:7.2f} | Storage: {storage_ratio:.0%} of original")

QR Decomposition
Any matrix A = Q @ R

- Q → Orthogonal matrix (Qᵀ @ Q = I, columns are orthonormal)
- R → Upper triangular matrix

ML uses: Numerically stable least squares, Gram-Schmidt orthogonalization,
computing eigenvalues iteratively (QR algorithm).

In [ ]:
# QR Decomposition — Base Python, NumPy, PyTorch, TF

A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 10]], dtype=float)

# NumPy
Q, R = np.linalg.qr(A)

print(f"Q (orthogonal, shape {Q.shape}):\n{np.round(Q, 4)}")
print(f"\nR (upper triangular, shape {R.shape}):\n{np.round(R, 4)}")
print(f"\nVerify Qᵀ @ Q = I:\n{np.round(Q.T @ Q, 4)}")
print(f"Reconstruct A = Q @ R matches: {np.allclose(A, Q @ R)}")

# PyTorch
Q_pt, R_pt = th.linalg.qr(th.tensor(A))
print(f"\nPyTorch QR — reconstruction matches: {th.allclose(th.tensor(A), Q_pt @ R_pt, atol=1e-5)}")

# TensorFlow
Q_tf, R_tf = tf.linalg.qr(tf.constant(A))
print(f"TensorFlow QR — reconstruction matches: {np.allclose(A, (Q_tf @ R_tf).numpy(), atol=1e-5)}")

Cholesky Decomposition
For any symmetric positive definite matrix A: A = L @ Lᵀ

- L → Lower triangular matrix (the "square root" of A)

ML uses: Gaussian Processes, Kalman filters,
sampling from multivariate Gaussian distributions (covariance matrices are always SPD).

In [ ]:
# Cholesky Decomposition

# A symmetric positive definite matrix (think: a covariance matrix)
A = np.array([[4, 2],
              [2, 3]], dtype=float)

# NumPy
L = np.linalg.cholesky(A)

print(f"A (symmetric positive definite):\n{A}")
print(f"\nL (lower triangular):\n{np.round(L, 4)}")
print(f"\nVerify L @ Lᵀ = A:\n{np.round(L @ L.T, 4)}")
print(f"Reconstruction matches: {np.allclose(A, L @ L.T)}")

# ML Example: Sample from a multivariate Gaussian N(μ, Σ)
# Trick: x = μ + L @ z, where z ~ N(0, I)
mean = np.array([0.0, 0.0])
covariance = np.array([[4.0, 2.0], [2.0, 3.0]])
L_cov = np.linalg.cholesky(covariance)

np.random.seed(0)
z = np.random.randn(2)           # standard normal
sample = mean + L_cov @ z        # transform to N(mean, cov)

print(f"\nML: Sampling from N(0, Σ) using Cholesky")
print(f"  Standard normal z:   {np.round(z, 4)}")
print(f"  Sample from N(0, Σ): {np.round(sample, 4)}")

# PyTorch
L_pt = th.linalg.cholesky(th.tensor(A))
print(f"\nPyTorch Cholesky — L @ Lᵀ = A: {th.allclose(L_pt @ L_pt.T, th.tensor(A), atol=1e-5)}")

# TensorFlow
L_tf = tf.linalg.cholesky(tf.constant(A))
print(f"TensorFlow Cholesky — L @ Lᵀ = A: {np.allclose((L_tf @ tf.transpose(L_tf)).numpy(), A, atol=1e-5)}")

10. Eigenvalues & Eigenvectors
For a square matrix A: A @ v = λ * v

- v → eigenvector: a special direction that A does NOT rotate — it only scales it
- λ → eigenvalue: how much A stretches (λ > 1), shrinks (λ < 1), or flips (λ < 0) that direction

ML uses:
- PCA → eigenvectors of the covariance matrix = principal components (directions of max variance)
- Spectral Clustering → eigenvectors of the graph Laplacian
- Google PageRank → dominant eigenvector of the link matrix
- Stability analysis → eigenvalues tell if a dynamical system converges or explodes

In [ ]:
# Base Python: Power Iteration — finds the dominant eigenvector
# Idea: repeatedly multiply A @ v and normalize.
# v converges to the eigenvector with the largest |eigenvalue|.

A = [[4, 2],
     [1, 3]]

def power_iteration(A, num_iterations=200):
    n = len(A)
    v = [1.0] * n
    for _ in range(num_iterations):
        Av = [sum(A[i][j] * v[j] for j in range(n)) for i in range(n)]
        norm = sum(x**2 for x in Av) ** 0.5
        v = [x / norm for x in Av]
    # Rayleigh quotient: λ = vᵀ A v
    Av = [sum(A[i][j] * v[j] for j in range(n)) for i in range(n)]
    eigenvalue = sum(v[i] * Av[i] for i in range(n))
    return eigenvalue, v

lam, vec = power_iteration(A)
print(f"Power Iteration — Dominant Eigenvalue: {lam:.4f}")
print(f"Power Iteration — Dominant Eigenvector: {[round(x, 4) for x in vec]}")

In [ ]:
# NumPy — eig (general), eigh (symmetric matrices, more stable)

A = np.array([[4, 2],
              [1, 3]], dtype=float)

eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Eigenvalues:  {np.round(eigenvalues, 4)}")
print(f"Eigenvectors (each column is one eigenvector):\n{np.round(eigenvectors, 4)}")

# Verify: A @ v = λ * v
for i in range(len(eigenvalues)):
    lam = eigenvalues[i]
    v   = eigenvectors[:, i]
    print(f"\nEigenvector {i+1}: v = {np.round(v, 4)}, λ = {lam:.4f}")
    print(f"  A @ v = {np.round(A @ v, 4)}")
    print(f"  λ * v = {np.round(lam * v, 4)}")
    print(f"  Match: {np.allclose(A @ v, lam * v)}")

In [ ]:
# PyTorch

A_pt = th.tensor([[4.0, 2.0],
                  [1.0, 3.0]])

eigenvalues_pt, eigenvectors_pt = th.linalg.eig(A_pt)

# For non-symmetric matrices eigenvalues may be complex — use .real for display
print(f"PyTorch Eigenvalues (real part): {eigenvalues_pt.real.tolist()}")
print(f"PyTorch Eigenvectors (real part):\n{eigenvectors_pt.real}")

In [ ]:
# TensorFlow

A_tf = tf.constant([[4.0, 2.0],
                    [1.0, 3.0]])

eigenvalues_tf, eigenvectors_tf = tf.linalg.eig(A_tf)

print(f"TF Eigenvalues (real part):  {eigenvalues_tf.numpy().real}")
print(f"TF Eigenvectors (real part):\n{eigenvectors_tf.numpy().real}")

In [ ]:
# ML Example: PCA from Scratch using Eigendecomposition
# PCA finds the directions (principal components) of maximum variance in data.

np.random.seed(42)
# Simulate 50 data points with 2 correlated features
X = np.random.randn(50, 2)
X[:, 1] = X[:, 0] * 0.8 + np.random.randn(50) * 0.3  # feature 2 is correlated with feature 1

# Step 1: Center the data (mean = 0)
X_centered = X - X.mean(axis=0)

# Step 2: Covariance matrix — captures how features co-vary
cov = (X_centered.T @ X_centered) / (len(X) - 1)  # (2x2)

# Step 3: Eigendecomposition of covariance (eigh for symmetric matrices)
eigenvalues, eigenvectors = np.linalg.eigh(cov)

# Step 4: Sort by eigenvalue descending (most important first)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Step 5: Explained variance ratio
explained = eigenvalues / eigenvalues.sum()

print(f"Covariance Matrix:\n{np.round(cov, 4)}")
print(f"\nEigenvalues (variance per component): {np.round(eigenvalues, 4)}")
print(f"Principal Components (columns):\n{np.round(eigenvectors, 4)}")
print(f"\nExplained Variance: PC1={explained[0]*100:.1f}%  PC2={explained[1]*100:.1f}%")
print(f"PC1 alone captures {explained[0]*100:.1f}% of all variance")

# Step 6: Project onto top-1 principal component (2D → 1D)
X_pca = X_centered @ eigenvectors[:, 0]  # (50,2) @ (2,) = (50,)
print(f"\nProjected data shape: {X.shape} → {X_pca.shape}")
print(f"First 5 projected values: {np.round(X_pca[:5], 4)}")

11. Rank
The rank of a matrix = number of linearly independent rows (or columns).
It tells you how much real, unique information the matrix carries.

- Full rank: all rows/columns are independent (no redundancy)
- Rank-deficient: some rows/columns are linear combinations of others (redundant data)

ML uses:
- Detecting redundant / collinear features in a dataset
- Checking if a linear system Ax = b has a unique solution (requires full rank)
- Low-rank approximations → SVD compression, LoRA fine-tuning of LLMs
- Understanding the effective capacity of a weight matrix

In [ ]:
# Base Python: Detect rank via Gaussian elimination
# Count how many pivot rows survive after eliminating dependent rows.

def matrix_rank(matrix):
    A = [row[:] for row in matrix]          # copy
    rows, cols = len(A), len(A[0])
    rank = 0
    for col in range(cols):
        pivot = next((r for r in range(rank, rows) if abs(A[r][col]) > 1e-9), None)
        if pivot is None:
            continue
        A[rank], A[pivot] = A[pivot], A[rank]
        for row in range(rank + 1, rows):
            factor = A[row][col] / A[rank][col]
            A[row] = [A[row][j] - factor * A[rank][j] for j in range(cols)]
        rank += 1
    return rank

# Full rank: 3 independent rows
A_full     = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
# row 3 = row 1 + row 2  → dependent
A_dep      = [[1, 2, 3], [4, 5, 6], [5, 7, 9]]
# all rows are multiples of [1, 2, 3]  → rank 1
A_rank1    = [[1, 2, 3], [2, 4, 6], [3, 6, 9]]

print(f"Full rank (3x3 identity):  rank = {matrix_rank(A_full)}")
print(f"Dependent row (3x3):       rank = {matrix_rank(A_dep)}")
print(f"All rows same direction:   rank = {matrix_rank(A_rank1)}")

In [ ]:
# NumPy, PyTorch, TensorFlow

A_full  = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=float)
A_dep   = np.array([[1, 2, 3], [4, 5, 6], [5, 7, 9]], dtype=float)  # row3 = row1 + row2
A_rank1 = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]], dtype=float)  # all rows proportional

print("--- NumPy ---")
print(f"Full rank (3×3):          rank = {np.linalg.matrix_rank(A_full)}")
print(f"Rank-deficient (3×3):     rank = {np.linalg.matrix_rank(A_dep)}")
print(f"Rank-1 matrix (3×3):      rank = {np.linalg.matrix_rank(A_rank1)}")

print("\n--- PyTorch ---")
print(f"Full rank:        rank = {th.linalg.matrix_rank(th.tensor(A_full)).item()}")
print(f"Rank-deficient:   rank = {th.linalg.matrix_rank(th.tensor(A_dep)).item()}")
print(f"Rank-1:           rank = {th.linalg.matrix_rank(th.tensor(A_rank1)).item()}")

print("\n--- TensorFlow ---")
print(f"Full rank:        rank = {tf.linalg.matrix_rank(tf.constant(A_full, dtype=tf.float32)).numpy()}")
print(f"Rank-deficient:   rank = {tf.linalg.matrix_rank(tf.constant(A_dep, dtype=tf.float32)).numpy()}")
print(f"Rank-1:           rank = {tf.linalg.matrix_rank(tf.constant(A_rank1, dtype=tf.float32)).numpy()}")

In [ ]:
# ML Example 1: Detecting Redundant Features (Collinearity)
# If rank(X) < n_features, some features carry no new information.

X_redundant = np.array([[1, 2, 3],
                        [2, 4, 6],
                        [3, 6, 9],
                        [4, 8, 12]], dtype=float)  # feat2 = 2*feat1, feat3 = 3*feat1

rank = np.linalg.matrix_rank(X_redundant)
n_features = X_redundant.shape[1]

print(f"Feature matrix shape:  {X_redundant.shape}")
print(f"Rank:                  {rank}  (out of {n_features} features)")
print(f"Redundant features:    {n_features - rank}")
print("→ Features 2 and 3 are multiples of Feature 1 — no additional information!")

print()

# ML Example 2: Low-Rank Approximation (LoRA concept for LLMs)
# Instead of storing full W (m×n), store A (m×r) and B (r×n) where r << min(m,n).
# This is exactly what LoRA does for fine-tuning large language models.

np.random.seed(0)
m, n, true_rank = 100, 80, 5

# Build a truly low-rank weight matrix
A_low = np.random.randn(m, true_rank)
B_low = np.random.randn(true_rank, n)
W = A_low @ B_low  # shape (100, 80), but rank = 5

print(f"Weight matrix W shape: {W.shape}")
print(f"True rank of W:        {np.linalg.matrix_rank(W)}")
print(f"Full storage:          {m * n:,} values")
print(f"Low-rank storage (r={true_rank}): {m*true_rank + true_rank*n:,} values")
print(f"Compression ratio:     {(m*true_rank + true_rank*n) / (m*n) * 100:.1f}% of original")

12. Determinant & Inverse — Deep Dive
We saw these briefly in Core Operations. Here we go deeper.

Determinant det(A):
- A scalar that measures how a linear transformation scales area/volume
- det > 0: preserves orientation | det < 0: flips orientation
- |det| > 1: stretches space | |det| < 1: shrinks space | |det| = 1: preserves area
- det = 0 → matrix is singular → no inverse exists (system Ax=b has no unique solution)

Inverse A⁻¹:
- Only exists when det(A) ≠ 0
- A @ A⁻¹ = A⁻¹ @ A = I
- Solves Ax = b → x = A⁻¹ @ b

Pseudo-inverse A⁺ (Moore-Penrose) — critical for ML:
- Works for any matrix (non-square, singular, underdetermined, overdetermined)
- Gives the least-squares solution to Ax = b
- Used in the Normal Equation for linear regression

In [ ]:
# Base Python: Determinant for 2×2 and 3×3

def det_2x2(A):
    # det([[a,b],[c,d]]) = a*d - b*c
    return A[0][0] * A[1][1] - A[0][1] * A[1][0]

def det_3x3(A):
    # Cofactor expansion along the first row
    return (A[0][0] * (A[1][1]*A[2][2] - A[1][2]*A[2][1])
          - A[0][1] * (A[1][0]*A[2][2] - A[1][2]*A[2][0])
          + A[0][2] * (A[1][0]*A[2][1] - A[1][1]*A[2][0]))

def inv_2x2(A):
    # A^{-1} = (1/det) * [[d,-b],[-c,a]]
    d = det_2x2(A)
    if abs(d) < 1e-12:
        return None  # singular — no inverse
    return [[A[1][1]/d, -A[0][1]/d],
            [-A[1][0]/d, A[0][0]/d]]

A        = [[3, 2], [1, 4]]
B        = [[1, 2, 3], [4, 5, 6], [7, 8, 10]]
singular = [[1, 2], [2, 4]]   # row 2 = 2 * row 1

print(f"det([[3,2],[1,4]]):       {det_2x2(A)}")
print(f"det(3×3 B):               {det_3x3(B)}")
print(f"det(singular [[1,2],[2,4]]): {det_2x2(singular)}  → no inverse!")
print(f"\nInverse of A:  {inv_2x2(A)}")
print(f"Inverse of singular: {inv_2x2(singular)}")

In [ ]:
# NumPy — det, inv, and pseudo-inverse (pinv)

A        = np.array([[3, 2], [1, 4]], dtype=float)
singular = np.array([[1, 2], [2, 4]], dtype=float)

print(f"det(A):        {np.linalg.det(A):.4f}")
print(f"det(singular): {np.linalg.det(singular):.4f}  → singular, no inverse")

inv_A = np.linalg.inv(A)
print(f"\nA⁻¹:\n{np.round(inv_A, 4)}")
print(f"A @ A⁻¹ = I:\n{np.round(A @ inv_A, 4)}")

# Pseudo-inverse: works for non-square and singular matrices
# Rectangular design matrix (4 samples, 2 params)
X = np.array([[1, 1.0],
              [1, 2.0],
              [1, 3.0],
              [1, 4.0]])

X_pinv = np.linalg.pinv(X)     # shape (2, 4)

print(f"\nX shape:             {X.shape}")
print(f"Pseudo-inverse shape: {X_pinv.shape}")
print(f"X⁺ @ X ≈ I:\n{np.round(X_pinv @ X, 4)}")

In [ ]:
# PyTorch — det, inv, pinv

A_pt        = th.tensor([[3.0, 2.0], [1.0, 4.0]])
singular_pt = th.tensor([[1.0, 2.0], [2.0, 4.0]])

print(f"PyTorch det(A):        {th.linalg.det(A_pt).item():.4f}")
print(f"PyTorch det(singular): {th.linalg.det(singular_pt).item():.4f}")

inv_A_pt = th.linalg.inv(A_pt)
print(f"\nPyTorch A⁻¹:\n{inv_A_pt}")
print(f"A @ A⁻¹ = I: {th.allclose(A_pt @ inv_A_pt, th.eye(2), atol=1e-5)}")

X_pt    = th.tensor([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0], [1.0, 4.0]])
X_pinv_pt = th.linalg.pinv(X_pt)
print(f"\nPyTorch pinv shape: {X_pinv_pt.shape}")
print(f"X⁺ @ X ≈ I: {th.allclose(X_pinv_pt @ X_pt, th.eye(2), atol=1e-5)}")

In [ ]:
# TensorFlow — det, inv, pinv

A_tf        = tf.constant([[3.0, 2.0], [1.0, 4.0]])
singular_tf = tf.constant([[1.0, 2.0], [2.0, 4.0]])

print(f"TF det(A):        {tf.linalg.det(A_tf).numpy():.4f}")
print(f"TF det(singular): {tf.linalg.det(singular_tf).numpy():.4f}")

inv_A_tf = tf.linalg.inv(A_tf)
print(f"\nTF A⁻¹:\n{inv_A_tf.numpy()}")
print(f"A @ A⁻¹ = I:\n{np.round((A_tf @ inv_A_tf).numpy(), 4)}")

X_tf     = tf.constant([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0], [1.0, 4.0]])
X_pinv_tf = tf.linalg.pinv(X_tf)
print(f"\nTF pinv shape: {X_pinv_tf.shape}")
print(f"X⁺ @ X ≈ I:\n{np.round((X_pinv_tf @ X_tf).numpy(), 4)}")

In [ ]:
# ML Example: Normal Equation — Analytical (closed-form) solution to Linear Regression
#
# Given: X @ w = y  (overdetermined: more samples than unknowns)
# Normal Equation:  w = (XᵀX)⁻¹ Xᵀy
# Pseudo-inverse:   w = X⁺ y   (same result, numerically more stable)

# House sizes (feature) and prices (target)
X = np.array([[1, 1.0],
              [1, 2.0],
              [1, 3.0],
              [1, 4.0],
              [1, 5.0]])    # first column = bias (all 1s)

y = np.array([2.0, 4.1, 5.9, 8.0, 10.1])  # roughly y = 2*size

# Normal Equation
XtX      = X.T @ X
Xty      = X.T @ y
w_normal = np.linalg.inv(XtX) @ Xty

# Pseudo-inverse shortcut
w_pinv   = np.linalg.pinv(X) @ y

print("Normal Equation (XᵀX)⁻¹Xᵀy:")
print(f"  w = [bias={w_normal[0]:.4f}, weight={w_normal[1]:.4f}]")

print("\nPseudo-inverse X⁺y:")
print(f"  w = [bias={w_pinv[0]:.4f},  weight={w_pinv[1]:.4f}]")

print(f"\nBoth methods give same result: {np.allclose(w_normal, w_pinv, atol=1e-6)}")

y_pred = X @ w_normal
print("\nPredictions vs Truth:")
for i in range(len(y)):
    print(f"  Predicted: {y_pred[i]:.2f}  |  Actual: {y[i]:.2f}")

print(f"\nMSE: {np.mean((y_pred - y)**2):.6f}")

Grand ML Example: Full PCA from Scratch
This ties everything together:

- Step 1: Center the data — vector subtraction
- Step 2: Covariance matrix — matrix multiplication, transpose
- Step 3: Eigendecomposition — eigenvalues & eigenvectors
- Step 4: Sort components — sort by eigenvalue magnitude
- Step 5: Project data — matrix multiplication
- Step 6: Reconstruct — transpose projection matrix
- Step 7: Verify — rank, orthogonality, SVD cross-check

In [ ]:
# Complete PCA from Scratch — tying all advanced topics together

np.random.seed(123)
n_samples, n_features = 100, 4

# Generate 4-feature data that actually lives in ~2 dimensions
Z = np.random.randn(n_samples, 2)           # 2 latent factors
W_gen = np.random.randn(2, n_features)      # mixing matrix
X = Z @ W_gen + 0.05 * np.random.randn(n_samples, n_features)

print(f"Data: {X.shape} ({n_samples} samples, {n_features} features)")
print(f"Observed rank of X: {np.linalg.matrix_rank(X)}  (signal has true rank ≈ 2)")

# --- Step 1: Center ---
X_mean = X.mean(axis=0)
X_c    = X - X_mean

# --- Step 2: Covariance matrix ---
cov = (X_c.T @ X_c) / (n_samples - 1)      # shape (4, 4)
print(f"\nCovariance matrix shape: {cov.shape}")
print(f"det(cov):  {np.linalg.det(cov):.6f}")
print(f"rank(cov): {np.linalg.matrix_rank(cov)}")

# --- Step 3: Eigendecomposition (eigh = symmetric, returns sorted ascending) ---
eigenvalues, eigenvectors = np.linalg.eigh(cov)

# --- Step 4: Sort descending ---
idx          = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

explained    = eigenvalues / eigenvalues.sum()
cumulative   = np.cumsum(explained)

print(f"\nEigenvalues:        {np.round(eigenvalues, 3)}")
print(f"Explained variance: {np.round(explained * 100, 1)}%")
print(f"Cumulative:         {np.round(cumulative * 100, 1)}%")
print(f"\nTop-2 PCs explain {cumulative[1]*100:.1f}% of all variance")

# --- Step 5: Projection matrix (top-2 PCs) ---
k = 2
P       = eigenvectors[:, :k]           # (4, 2) — orthonormal columns
X_pca   = X_c @ P                      # (100, 4) @ (4, 2) = (100, 2)

print(f"\nProjection matrix P shape: {P.shape}")
print(f"Projected data shape: {X_pca.shape}")
print(f"Pᵀ @ P = I (orthonormal): {np.allclose(P.T @ P, np.eye(k), atol=1e-10)}")

# --- Step 6: Reconstruct (lossy) ---
X_recon = X_pca @ P.T + X_mean         # (100, 2) @ (2, 4) = (100, 4)
mse     = np.mean((X - X_recon) ** 2)

print(f"\nReconstruction MSE (top-{k} of {n_features} components): {mse:.6f}")

# Compare with SVD-based PCA (should give identical results)
U_svd, s_svd, Vt_svd = np.linalg.svd(X_c, full_matrices=False)
X_pca_svd = X_c @ Vt_svd[:k].T

# Eigenvector signs can differ — compare absolute projections
print(f"Eigendecomp == SVD-based PCA: {np.allclose(np.abs(X_pca), np.abs(X_pca_svd), atol=1e-8)}")
print("\nPCA complete: dimensionality reduced from 4 → 2 with minimal information loss.")